# 📓 Semana 5 · Dia 3 — Lakeflow pipelines (Delta Live Tables)

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA, DEP (Lakeflow) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Pipeline DLT Bronze→Prata rodando |

---


## 📖 Teoria — O que é o Lakeflow (DLT)

**Delta Live Tables (DLT)** é o framework declarativo do Databricks para pipelines: você declara **o que** cada tabela deve ser, e o DLT gerencia dependências, ordenação, qualidade e retry.

Na Free Edition: **1 pipeline ativo por tipo** (crie e rode; pare antes de criar outro).

Sintaxe: decorators `@dlt.table`, `@dlt.view`, `@dlt.expect` (e `@dlt.streaming_table`).


## 📖 Teoria — Materialized vs Streaming tables

**Materialized table (MT)**: calculada como batch, recalculada sob demanda — refrescada de acordo com as dependências.
**Streaming table (ST)**: alimentada por stream contínuo (Auto Loader) com incrementos.

Em produção, usa-se ST para ingestão (Bronze) e MT para transformações (Prata/Ouro) — o DLT orquestra tudo.


### 💻 Na prática — Pipeline DLT em arquivo

Crie o arquivo de pipeline como *workspace file* e rode via UI. Células abaixo são o conteúdo do arquivo — cole num arquivo `.py` no Workspace.


In [ ]:
# ===== workspace_file: pipeline_vendas.py =====
import dlt
from pyspark.sql.functions import col, to_date, sum as s

@dlt.table(comment="Bronze: vendas via Auto Loader")
def vendas_bronze():
    return (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/vol_checkpoints/schema_pipeline")
        .option("header", True)
        .load("/Volumes/workspace/bronze/vol_landing"))

@dlt.table(comment="Prata: limpo e tipado")
def vendas_prata():
    return (dlt.read_stream("vendas_bronze")
        .filter(col("Quantity") > 0)
        .withColumn("data_venda", to_date("InvoiceDate", "M/d/yyyy H:mm")))

@dlt.table(comment="Ouro: receita por dia")
def receita_diaria():
    return (dlt.read("vendas_prata")
        .groupBy("data_venda")
        .agg(s(col("Quantity") * col("UnitPrice")).alias("receita")))

### 💻 Na prática — Rodando o pipeline

1. Em **Workflows → Pipelines → Create Pipeline**.
2. Nome: `pipeline_vendas`. Selecione o arquivo acima como código-fonte.
3. Target schema: `workspace.bronze` (ou um schema próprio).
4. **Start**. O DLT cria as tabelas e mostra o DAG de dependências.
5. Na Free Edition, pare o pipeline ao terminar (limite de 1 ativo).


In [ ]:
# Conferir o resultado após rodar o pipeline
spark.sql("SHOW TABLES IN workspace.bronze")
display(spark.sql("SELECT * FROM workspace.bronze.receita_diaria LIMIT 10"))

> 🎯 **Dica de prova**: DLT cai forte na DEA/DEP 2026 (nomenclatura: **Lakeflow pipelines**). Decore: `@dlt.table`, `@dlt.view`, `@dlt.expect`, `dlt.read` vs `dlt.read_stream`, e os 3 níveis de expectations (próximo dia).


## 🎯 Exercícios de fixação

**1.** Qual a diferença entre dlt.read e dlt.read_stream?

**2.** Por que o DLT é declarativo e o notebook é imperativo?

**3.** Crie uma tabela DLT extra que compute top 10 produtos por dia.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** read vs read_stream

`dlt.read` lê a versão materializada (batch) de outra tabela; `dlt.read_stream` lê como stream contínuo — usado em streaming tables.

**2.** Declarativo vs imperativo

No DLT você declara o RESULTADO (o que é a tabela); o DLT decide ordem, incrementos, retry e qualidade. No notebook você dita o COMO passo a passo.

**3.** Top 10

```python
@dlt.table
def top10_dia():
    return (dlt.read('vendas_prata')
        .groupBy('data_venda','StockCode')
        .agg(s(col('Quantity')).alias('qtd'))
        .orderBy('data_venda', col('qtd').desc()))
```



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*